## Этап 1 - Загрузка исходного parquet и результата модели

In [9]:
import pandas as pd 
from pathlib import Path

In [10]:
DATA_INPUT_ORIGINAL = Path("../data/interim") / "merged_buildings_by_geometry.parquet"
DATA_INPUT_RESULT = Path("../data/interim") / "merged_buildings_filled_with_catboost.parquet"

df_original = pd.read_parquet(DATA_INPUT_ORIGINAL)
df_result = pd.read_parquet(DATA_INPUT_RESULT)

In [11]:
df_model = df_result[[
    "component_id",
    "target_height_predicted",
    "target_height_filled",
    "target_height_fill_source",
    "target_height_was_predicted",
]].copy()

print("Исходный df:", df_original.shape)
print("Модельный df:", df_model.shape)

Исходный df: (139649, 48)
Модельный df: (139649, 5)


In [12]:
print("Дубликаты в df_original:", df_original["component_id"].duplicated().sum())
print("Дубликаты в df_model:", df_model["component_id"].duplicated().sum())

print("Уникальных id в original:", df_original["component_id"].nunique())
print("Уникальных id в model:", df_model["component_id"].nunique())

Дубликаты в df_original: 0
Дубликаты в df_model: 0
Уникальных id в original: 139649
Уникальных id в model: 139649


## Этап 2 - слияние

In [13]:
df_merged = df_original.merge(
    df_model,
    on="component_id",
    how="left",
)

print("После merge:", df_merged.shape)

После merge: (139649, 52)


## Этап 3 - Проверка

In [14]:
print("Предсказанных высот:", df_merged["target_height_was_predicted"].sum())
print("NaN в predicted:", df_merged["target_height_predicted"].isna().sum())
print("NaN в filled:", df_merged["target_height_filled"].isna().sum())

df_merged[[
    "component_id",
    "target_height",
    "target_height_predicted",
    "target_height_filled",
    "target_height_fill_source",
]].head(10)

Предсказанных высот: 38086
NaN в predicted: 101563
NaN в filled: 0


,component_id,target_height,target_height_predicted,target_height_filled,target_height_fill_source
0,1,4.50,NaN,4.500000,observed
1,2,4.50,NaN,4.500000,observed
2,3,60.00,NaN,60.000000,observed
3,4,NaN,3.423235,3.423235,catboost
4,5,NaN,3.423124,3.423124,catboost
5,6,48.00,NaN,48.000000,observed
6,7,68.25,NaN,68.250000,observed
7,8,NaN,3.465983,3.465983,catboost
8,9,NaN,3.460535,3.460535,catboost
9,10,NaN,3.470120,3.470120,catboost


In [16]:
FINAL_PATH = Path("../data/processed/merged_buildings_final_with_heights.parquet")

df_merged.to_parquet(FINAL_PATH, index=False)

print(f"Файл сохранен: {FINAL_PATH}")

Файл сохранен: ..\data\processed\merged_buildings_final_with_heights.parquet
